# 1 - Entrenamiento del modelo
Transfer learning con EfficientNet-B0 preentrenado en ImageNet para clasificar 38 enfermedades en plantas.

# 2 - Imports


In [1]:
import os
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

# verifica GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

Usando: cpu


# 3 - Transformacion de imagenes
Preparamos las imágenes para que el modelo las entienda: las redimensionamos, 
aplicamos augmentación y normalizamos con los valores estándar de ImageNet.

In [2]:
DATASET_PATH = "plantvillage dataset/color"
IMG_SIZE = 224      # tamaño estándar que espera EfficientNet
BATCH_SIZE = 32     # imágenes que procesa a la vez
NUM_CLASSES = 38    # total de enfermedades

# transformaciones para entrenamiento (con augmentación)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),        # voltea imagen horizontalmente al azar
    transforms.RandomRotation(15),            # rota hasta 15 grados al azar
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # varía brillo y contraste
    transforms.ToTensor(),                    # convierte imagen a tensor numérico
    transforms.Normalize(                     # normaliza con valores de ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# transformaciones para validación y test (sin augmentación)
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transformaciones definidas correctamente")

Transformaciones definidas correctamente


# 4 - DataLoaders
Dividimos el dataset en 80% entrenamiento, 10% validación y 10% test.
Luego creamos los DataLoaders que alimentan las imágenes al modelo en batches.

In [ ]:
# carga todo el dataset desde la carpeta
dataset_completo = datasets.ImageFolder(root=DATASET_PATH, transform=train_transforms)

# calcula tamaños de cada split
total = len(dataset_completo)
train_size = int(0.8 * total)
validation_size = int(0.1 * total)
test_size = total - train_size - validation_size

print(f"Total imágenes:      {total:,}")
print(f"Entrenamiento:       {train_size:,}")
print(f"Validación:          {validation_size:,}")
print(f"Test:                {test_size:,}")

# divide aleatoriamente
train_dataset, val_dataset, test_dataset = random_split(
    dataset_completo,
    [train_size, validation_size, test_size],
    generator=torch.Generator().manual_seed(42)  # seed fija = resultados reproducibles
)

# aplica transformaciones de validación al val y test
val_dataset.dataset.transform = val_transforms
test_dataset.dataset.transform = val_transforms

# crea los DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nDataLoaders creados correctamente")
print(f"Clases: {dataset_completo.classes[:3]}...")

Total imágenes:      54,305
Entrenamiento:       43,444
Validación:          5,430
Test:                5,431

DataLoaders creados correctamente
Clases: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']...
